In [3]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
import math

In [6]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol without an attacker.

# Initialize the simulator
simulator = AerSimulator()

def quantum_rng(num_bits):
    """Generates random bits by measuring a qubit in superposition."""
    qc = QuantumCircuit(num_bits, num_bits)
    qc.h(range(num_bits)) # Put all qubits in superposition
    qc.measure(range(num_bits), range(num_bits))

    result = simulator.run(qc, shots=1).result()
    bitstring = list(result.get_counts().keys())[0]
    # Reverse because Qiskit reads right-to-left
    return [int(b) for b in bitstring[::-1]]

def encode_qubits(bits, bases):
    """Encodes bits into quantum states."""
    circuits = []
    for i in range(len(bits)):
        qc = QuantumCircuit(1, 1)
        if bits[i] == 1:
            qc.x(0) # Apply X gate for bit 1
        if bases[i] == 1:
            qc.h(0) # Apply H gate for Diagonal basis
        circuits.append(qc)
    return circuits

def measure_qubits(circuits, bases):
    """Measures qubits using specified bases."""
    measured_bits = []
    for i in range(len(circuits)):
        qc = circuits[i]
        if bases[i] == 1:
            qc.h(0) # Apply H gate to measure in Diagonal basis
        qc.measure(0, 0)

        result = simulator.run(qc, shots=1).result()
        bit = int(list(result.get_counts().keys())[0])
        measured_bits.append(bit)
    return measured_bits

# ==========================================
# SIMULATION: BB84 PLAIN
# ==========================================
num_qubits = 50

# --- ALICE ---
alice_bits = quantum_rng(num_qubits)
alice_bases = quantum_rng(num_qubits)
qubits_to_send = encode_qubits(alice_bits, alice_bases)

# --- BOB ---
bob_bases = quantum_rng(num_qubits)
bob_bits = measure_qubits(qubits_to_send, bob_bases)

# --- PUBLIC DISCUSSION (SIFTING) ---
alice_key = []
bob_key = []

for i in range(num_qubits):
    if alice_bases[i] == bob_bases[i]:
        alice_key.append(alice_bits[i])
        bob_key.append(bob_bits[i])

# --- RESULTS ---
print("--- PROTOCOL RESULTS ---")
print(f"Initial Qubits Sent: {num_qubits}")
print(f"Sifted Key Length:   {len(alice_key)}")
print(f"Alice's Key:         {alice_key}")
print(f"Bob's Key:           {bob_key}")

# Verify no errors
errors = sum(1 for a, b in zip(alice_key, bob_key) if a != b)
print(f"Total Errors:        {errors}")
if errors == 0:
    print("Success: Alice and Bob share the exact same secure key.")
else:
  print("Failure: Alice and Bob do not share teh exact same secure key")

--- PROTOCOL RESULTS ---
Initial Qubits Sent: 50
Sifted Key Length:   26
Alice's Key:         [0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0]
Bob's Key:           [0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0]
Total Errors:        0
Success: Alice and Bob share the exact same secure key.
